In [ ]:
# import sckit-learn, TensorFlow, and Keras, pandas and matplotlib and numpy
import sklearn
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
#load the fashion MNIST dataset
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
print(X_train_full.shape)
print(X_train_full.dtype)
# ----don't use a validation set---
X_train = X_train_full / 255.0
y_train = y_train_full
# ----use a validation set----
# X_valid, X_train = X_train_full[:5000] / 255.0, X_train_full[5000:] / 255.0
# y_valid, y_train = y_train_full[:5000], y_train_full[5000:]
X_test = X_test / 255.0
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]


In [ ]:
# example of one item in the dataset
class_names[y_train[0]]
plt.imshow(X_train[0], cmap="binary")
plt.axis("off")
plt.show()

In [ ]:
layer = tf.keras.layers.Dense(100, activation="relu",
                              kernel_initializer="he_normal",
                              kernel_regularizer=tf.keras.regularizers.l2(0.01))

tf.random.set_seed(42)  # extra code – for reproducibility

In [ ]:
from functools import partial

RegularizedDense = partial(tf.keras.layers.Dense,
                           activation="relu",
                           kernel_initializer="he_normal",
                           kernel_regularizer=tf.keras.regularizers.l2(0.01))

model_regularized = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    RegularizedDense(100),
    RegularizedDense(100),
    RegularizedDense(10, activation="softmax",
                     kernel_initializer="glorot_normal")  # not in the book
])

In [ ]:
# extra code – compile and train the model
optimizer = tf.keras.optimizers.SGD(learning_rate=0.02)
model_regularized.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])
history_regularized = model_regularized.fit(X_train, y_train, epochs=2,                                       
                    validation_data=(X_test, y_test))

In [ ]:
tf.random.set_seed(42)  # extra code – for reproducibility

In [ ]:
model_dropout = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(10, activation="softmax")
])

In [ ]:
# extra code – compile and train the model
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
model_dropout.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])
history_dropout = model_dropout.fit(X_train, y_train, epochs=10,
                   validation_data=(X_test, y_test))

In [ ]:
print(model_dropout.evaluate(X_train, y_train))
print(model_regularized.evaluate(X_train, y_train))

In [ ]:
print(model_dropout.evaluate(X_test, y_test))
print(model_regularized.evaluate(X_test, y_test))

In [ ]:
# plot dropout and regularization training history accuracy
epochs_dropout = range(1, len(history_dropout.history["accuracy"]) + 1)
epochs_regularized = range(1, len(history_regularized.history["accuracy"]) + 1)
plt.plot(epochs_dropout, history_dropout.history["val_accuracy"], "b--", label="dropout test")
plt.plot(epochs_regularized, history_regularized.history["val_accuracy"], "orange", linestyle="--", label="regularization test")
plt.plot(epochs_dropout, history_dropout.history["accuracy"], label="dropout")
plt.plot(epochs_regularized, history_regularized.history["accuracy"], label="regularization")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()